In [1]:
# Python 3.10.11
# %pip install -r requirements.txt > /dev/null
from args import *
from utils import *

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from torchmetrics import AUROC
from torch.utils.tensorboard import SummaryWriter


# TODO(241225) 依赖导入
import pandas as pd


In [8]:
# TODO(241225) 导入label数据
label_df = pd.read_csv(sample_labels_file_path, index_col=0)

# 裁减样本数量，使得 0 - 1 样本数量一致
df_0 = label_df[label_df['label'] == 0]
df_1 = label_df[label_df['label'] == 1]
min_len = min(len(df_0), len(df_1))
new_df = pd.concat([df_0[:min_len], df_1[:min_len]], axis=0)

sample_key_list = label_df.index.to_list()
sample_key_list = new_df.index.to_list()

print(f"label 样本数量: {len(sample_key_list)}")

# TODO(241225) 导入gene数据
# 根据 sample_key_list 为基准, 若模态数据中不存在 sample_key 则填充新数据
gene_array = load_gene_data_by_sample_key(sample_key_list).values
cnv_array = load_cnv_data_by_sample_key(sample_key_list).values

wsi_array = load_wsi_data_by_sample_key(sample_key_list)
report_array = load_report_data_by_sample_key(sample_key_list)

label_array = label_df.values
label_array = new_df.values

_, gene_dim = gene_array.shape
_, cnv_dim = cnv_array.shape
_, wsi_dim = wsi_array.shape
_, report_dim = report_array.shape
_, label_dim = label_array.shape

print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

batch_size = 32

all_dataset = MultiOmicsDataset(gene_array, cnv_array, report_array, wsi_array, label_array)

# # 假设 all_dataset 是一个 Dataset 对象
# train_len = int(len(all_dataset) * 0.8)  # 80% 的数据用作训练集
# test_len = len(all_dataset) - train_len  # 剩余的数据用作验证集

# # 使用 random_split 分割数据集
# train_val_dataset, test_dataset = random_split(all_dataset, [train_len, test_len])

train_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)
val_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)

label 样本数量: 152

gene 数据维度:   2340
cnv 数据维度:    2340
wsi 数据维度:    2048
report 数据维度: 768



In [9]:
### 分割线

In [10]:

from torch import nn, optim

def report_loader_layer(pre_trained=False):

    if not pre_trained:
        return nn.Sequential(
            nn.Linear(768, 768),
            nn.Linear(768, 2),
        )



    model_file_path = pkg_dir_path.parent / "report/data/output/bert.pth"
    assert model_file_path.exists()

    model = torch.load(model_file_path).to(device)
    model = model.classifier

    # 冻结参数
    # 冻结第一层
    for param in model[0].parameters():
        param.requires_grad = False

    # 确认第一层的参数不需要梯度
    for name, param in model.named_parameters():
        if name.startswith('0.'):
            print(name, param.requires_grad)  # 应该输出False

    return model

report_loader_layer = report_loader_layer()

In [11]:

from torch import nn, optim

def graphy_loader_layer(pre_trained=False):

    if not pre_trained:
        return nn.Sequential(
            nn.Linear(2048, 2048),
            nn.Linear(2048, 2),
        )

    return model

graphy_loader_layer = graphy_loader_layer()

In [12]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import models  # 如果需要使用预训练模型

# 假设我们使用一个预训练的ResNet作为特征提取器，并添加自定义层
class CustomModel(nn.Module):
    def __init__(self, base_model, num_classes):
        super(CustomModel, self).__init__()
        # 假设base_model期望的输入形状是[batch_size, 3, 224, 224]
        # 我们将第一层替换为一个全连接层
        self.base_model = base_model
        # self.custom_layer = nn.Sequential(
        #     nn.Linear(base_model.in_features, 256),
        #     nn.ReLU(),
        #     # nn.Dropout(0.5),
        #     nn.Linear(256, num_classes)
        # )

    def forward(self, x):
        x = self.base_model(x)
        # x = self.custom_layer(x)
        return x

# 加载预训练模型并替换最后一层
num_classes = 1  # 根据您的任务确定类别数
model = CustomModel(report_loader_layer, num_classes).to(device)
model = CustomModel(graphy_loader_layer, num_classes).to(device)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 适用于二分类问题
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 学习率调度器，可以在训练过程中调整学习率
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')

In [13]:
from sklearn.metrics import roc_auc_score

In [15]:
# 初始化TensorBoard SummaryWriter
writer = SummaryWriter(data_output_dir_path / 'runs/multi-model/graphy')

num_epochs = 100
for epoch in range(num_epochs):

    model.train()
    # 导入 batch 数据
    for batch in train_loader:
        gene_tensor = batch["gene_tensor"].to(device)
        cnv_tensor = batch["cnv_tensor"].to(device)
        report_tensor = batch["report_tensor"].to(device)
        wsi_tensor = batch["wsi_tensor"].to(device)
        label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
        label_tensor = label_tensor.squeeze()
        label_tensor = label_tensor.long()

        # outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)
        outputs = model(wsi_tensor)  # 假设wsi_tensor是主要的输入特征
        # print((outputs.shape, label_tensor.shape))

        optimizer.zero_grad()
        loss = criterion(outputs, label_tensor)
        loss.backward()
        optimizer.step()

    print("train loss: ", loss.item())
    writer.add_scalar('training_loss', loss.item(), epoch)

    correct = 0
    total = 0
    # 初始化变量来存储所有预测值和标签
    all_labels = []
    all_preds_prob = []

    # 在每个epoch结束时进行验证
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            gene_tensor = batch["gene_tensor"].to(device)
            cnv_tensor = batch["cnv_tensor"].to(device)
            report_tensor = batch["report_tensor"].to(device)
            wsi_tensor = batch["wsi_tensor"].to(device)
            label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
            label_tensor = label_tensor.squeeze()
            label_tensor = label_tensor.long()

            outputs = model(wsi_tensor)

            _, preds = torch.max(outputs, 1)  # 获取预测的类别
            total += label_tensor.size(0)
            correct += (preds == label_tensor).sum().item()
            
            # 假设outputs是模型的输出，形状为[24, 2]
            # 我们只关心正类的概率，所以取第二列（索引为1）
            all_preds_prob.extend(torch.sigmoid(outputs)[:, 1].cpu().numpy())
            all_labels.extend(label_tensor.cpu().numpy())

    # 计算准确率
    accuracy = correct / total
    print(f"Validation Accuracy: ({accuracy})")
    writer.add_scalar('validation_accuracy', accuracy, epoch)

    # 计算AUC
    auc = roc_auc_score(all_labels, all_preds_prob)
    print(f'AUC: {auc}')
    writer.add_scalar('validation_auc', auc, epoch)

writer.close()

train loss:  1.0853753089904785
Validation Accuracy: (0.5131578947368421)
AUC: 0.5604224376731302
train loss:  0.6881751418113708
Validation Accuracy: (0.6842105263157895)
AUC: 0.7553670360110802
train loss:  0.4719879627227783
Validation Accuracy: (0.7894736842105263)
AUC: 0.912915512465374
train loss:  0.41348400712013245
Validation Accuracy: (0.8421052631578947)
AUC: 0.9634695290858726
train loss:  0.27816328406333923
Validation Accuracy: (0.8486842105263158)
AUC: 0.9605263157894737
train loss:  0.2989520728588104
Validation Accuracy: (0.9407894736842105)
AUC: 0.9875346260387812
train loss:  0.12368255108594894
Validation Accuracy: (0.9473684210526315)
AUC: 0.9987880886426593
train loss:  0.08693915605545044
Validation Accuracy: (0.9868421052631579)
AUC: 0.9996537396121884
train loss:  0.10847082734107971
Validation Accuracy: (0.993421052631579)
AUC: 0.9998268698060943
train loss:  0.042064979672431946
Validation Accuracy: (0.993421052631579)
AUC: 0.9998268698060943
train loss:  0.0